# Track 3 — Smart Farm (Environmental)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** sustainability- or life-sciences-curious students.
**Data:** real Singapore rainfall (NEA, 5-minute, island-wide) and PSI/PM2.5
(hourly, 5 regions), pulled from `data.gov.sg`.
**Macro lesson:** *Forecasting + threshold decisions — "should we water tomorrow?"*
**Your built-in baseline:** "do whatever yesterday suggests", or "always water on
Wednesdays".

---
This is **real data with real gaps**. Singapore rainfall is extremely spiky —
most days are zero and a handful are enormous. That skew is the whole challenge
of this track, and it is why a naive average is a poor guide to tomorrow.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.


In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['sg_psi_hourly.csv', 'sg_rainfall_daily.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track3_smart_farm/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os

CSV = "sg_rainfall_daily.csv"
if not os.path.exists(CSV):
    from google.colab import files; files.upload()      # pick the CSV from the pack

rain = pd.read_csv(CSV, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
psi  = pd.read_csv("sg_psi_hourly.csv", parse_dates=["timestamp"]) \
         if os.path.exists("sg_psi_hourly.csv") else None

In [ ]:
# --- FIRST LOOK -------------------------------------------------------
print("rows:", len(rain), " range:", rain.date.min().date(), "->", rain.date.max().date())
print("\nisland-wide mean daily rainfall (mm), averaged over 77 NEA stations:")
print(rain.rain_mm_mean.describe().round(2).to_string())
print(f"\ncompletely dry days (<0.2mm) : {(rain.rain_mm_mean < 0.2).mean():.0%}")
print(f"days below the 1mm threshold : {(rain.rain_mm_mean < 1.0).mean():.0%}")
print("\n>>> Note how far the max is from the median. Singapore rainfall is")
print(">>> heavily right-skewed: many small days, a few enormous ones. The mean")
print(">>> is NOT a typical day, so be careful using it as a prediction.")
rain.head()

In [ ]:
# --- LIVE REFRESH (optional) -----------------------------------------
# The CSV above was cached so labs don't depend on the network. To pull
# today's readings live from data.gov.sg, run this. No API key needed.
# NOTE: the API returns 25 readings per page, so a full day = ~12 requests.

# import requests
# s = requests.Session(); s.headers.update({"User-Agent": "EP0418-student"})
# rows, tok = [], None
# while True:
#     p = {"date": "2026-07-24"}
#     if tok: p["paginationToken"] = tok
#     d = s.get("https://api-open.data.gov.sg/v2/real-time/api/rainfall",
#               params=p, timeout=30).json()["data"]
#     for rd in d["readings"]:
#         for item in rd["data"]:
#             rows.append((rd["timestamp"], item["stationId"], item["value"]))
#     tok = d.get("paginationToken")
#     if not tok: break
# live = pd.DataFrame(rows, columns=["timestamp", "station_id", "rain_mm"])
# print(live.shape)

In [ ]:
# --- PLOT -------------------------------------------------------------
fig, ax = plt.subplots(2, 1, figsize=(13, 7))
ax[0].bar(rain.date, rain.rain_mm_mean, width=.9, color="tab:blue")
ax[0].set_title("Island-wide mean daily rainfall (NEA)"); ax[0].set_ylabel("mm")
ax[0].grid(alpha=.3)
ax[1].plot(rain.date, rain.rain_mm_mean.rolling(7).mean(), color="tab:red", lw=1.8)
ax[1].set_title("7-day rolling mean — the trend under the spikes")
ax[1].set_ylabel("mm"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- YOUR BASELINE: four irrigation policies --------------------------
# Decision each day: water the field, or don't?
# Rule: the field NEEDS water if less than 1mm of rain fell that day.
NEEDS = rain.rain_mm_mean < 1.0

policies = {
    "A: yesterday's value": NEEDS.shift(1, fill_value=True),
    "B: always Wednesday":  rain.date.dt.dayofweek == 2,
    "C: never water":       pd.Series(False, index=rain.index),
    "D: always water":      pd.Series(True,  index=rain.index),
}
print(f"{'policy':22s} {'correct':>8}  {'missed':>7}  {'wasted':>7}")
for name, pol in policies.items():
    print(f"{name:22s} {(pol == NEEDS).mean():7.1%}  "
          f"{(NEEDS & ~pol).sum():7d}  {(~NEEDS & pol).sum():7d}")

print("\nlag-1 autocorrelation of daily rainfall:", round(rain.rain_mm_mean.autocorr(1), 3))

### Read those four rows carefully — they contain the whole lesson

**"Never water" has the best accuracy of the four.** It is also an irrigation policy
that lets the crop die on all 26 days the field needed water. Accuracy is the wrong
scoreboard whenever the two errors cost different amounts, and here they cost wildly
different amounts (SLO 6.4, 7.1).

**"Yesterday's value" is the worst of the four.** Look at the lag-1 autocorrelation
printed above — it is close to zero and slightly *negative*. Whether it rained today
tells you essentially nothing about tomorrow in Singapore's convective climate. A
persistence baseline needs the series to have memory, and this one does not.

So your job is *not* to maximise accuracy. It is to state what a missed watering costs
versus a wasted one, then choose the threshold that minimises **that** cost. Two
students with different cost assumptions should defensibly build different systems.

---
## Three macro questions to investigate

**1. What is your forecast actually *for*?**
You are not predicting rainfall for its own sake — you are deciding whether to
turn on a pump. A forecast that is 2mm off matters enormously at the 1mm decision
boundary and not at all at 40mm. Score your model on **decisions it got right**,
not only on MAE/RMSE (SLO 6.4, 7.1).

**2. Does rain tomorrow depend on rain today?**
Plot rainfall against itself lagged by one day. If the cloud is shapeless, daily
rainfall is close to memoryless and Prophet will struggle to beat "assume dry" —
which is a real and reportable result (SLO 6.2). Try weekly totals instead:
aggregation often reveals structure that daily noise hides.

**3. Should PSI change your irrigation decision at all?**
You have PM2.5 alongside rainfall. Haze and rain are related — but does knowing
today's PSI improve *tomorrow's* rainfall forecast? Test it honestly. Adding a
feature that does not help, and saying so, is better work than adding one that
looks impressive.

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns `date, rain_mm_mean, rain_mm_max, stations_reporting`.
> The time column is `date`. Fit a Prophet model to daily rainfall, forecast 14 days ahead, and compare its MAE against a seasonal-naive baseline that predicts the same weekday from last week.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".